# 05 Retrieval - Sample Or Full Index Evaluation

Sample mode builds FAISS indexes over sample embeddings. Full mode builds indexes over the server-scale embeddings.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd

sys.path.append('..')
from src.full_pipeline import (
    PROCESSED_PATH,
    EMBEDDING_DIR,
    EXPERIMENTS_DIR,
    parquet_row_count,
    sample_processed_path,
    sample_embedding_dir,
    run_full_retrieval_for_embeddings,
)
from src.visualize import plot_metrics_comparison


## 1. Retrieval Configuration


In [ ]:
RUN_MODE = "sample"  # "sample" for local, "full" for server
ROWS_PER_CATEGORY = 1_000

ACTIVE_PROCESSED_PATH = sample_processed_path(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else PROCESSED_PATH
ACTIVE_EMBEDDING_DIR = sample_embedding_dir(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else EMBEDDING_DIR
ACTIVE_EXPERIMENTS_DIR = EXPERIMENTS_DIR / 'local_sample' if RUN_MODE == "sample" else EXPERIMENTS_DIR

ENCODERS = ['tfidf', 'w2v', 'glove', 'sbert', 'bge']
LABEL_COLUMN = 'category'
QUERY_SAMPLE_SIZE = 1_000 if RUN_MODE == "sample" else 5_000
K_VALUES = [5, 10, 50]
MIN_EXPECTED_ROWS = 100 if RUN_MODE == "sample" else 1_000_000

if not ACTIVE_PROCESSED_PATH.exists():
    raise FileNotFoundError(f"{ACTIVE_PROCESSED_PATH} not found. Run 02_preprocess.ipynb first.")
row_count = parquet_row_count(ACTIVE_PROCESSED_PATH)
if row_count < MIN_EXPECTED_ROWS:
    raise RuntimeError(f"{ACTIVE_PROCESSED_PATH} has only {row_count:,} rows. Run 02_preprocess.ipynb first.")

print(f"Run mode: {RUN_MODE}; rows: {row_count:,}; embedding dir: {ACTIVE_EMBEDDING_DIR}")


## 2. Build FAISS Indexes And Evaluate Queries


In [ ]:
all_metrics = run_full_retrieval_for_embeddings(
    processed_path=ACTIVE_PROCESSED_PATH,
    embedding_dir=ACTIVE_EMBEDDING_DIR,
    results_dir=ACTIVE_EXPERIMENTS_DIR / 'retrieval',
    encoders=ENCODERS,
    label_column=LABEL_COLUMN,
    query_sample_size=QUERY_SAMPLE_SIZE,
    k_values=K_VALUES,
)

display(pd.DataFrame(all_metrics).T)


## 3. Retrieval Metric Comparison


In [ ]:
if all_metrics:
    plot_metrics_comparison(
        all_metrics,
        filename=str(ACTIVE_EXPERIMENTS_DIR / 'figures' / 'retrieval_metrics_comparison.png'),
    )
else:
    print("No retrieval metrics generated. Run 03_encode.ipynb first.")
